## Module 7: Overfitting Regularization

In [2]:
# L2 Regularization (weight_decay)
import torch.optim as optim
optimizer = optim.Adam(model.parameters(), lr = 0.001, weight_decay = 1e-4)

# AdamW applies weight decay correctly
optimizer = optim.AdamW(model.parameters(), lr = 0.001, weight_decay = 1e-4)


# L1 Regularization (Lasso)
def l1_penalty(model, lambda_l1):
    l1_sum = 0.0
    for name, param in model.named_parameters():
        if 'bias' not in nme:
            l1_sum += param.abs().sum()
    return lambda_l1 * l1_sum

loss = criterion(pred, y) + l1_penalty(model, lambda_l1 = 1e-4)

NameError: name 'model' is not defined

In [3]:
# Dropout
import torch
import torch.nn as nn

class ModelWithDropout(nn.Module):
    def __init__(self, in_features, p_drop = 0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 256), nn.ReLU(), nn.Dropout(p=p_drop), nn.Linear(256, 128), 
            nn.ReLU(), nn.Dropout(p=p_drop), nn.Linear(128, 64), nn.ReLU(), nn.Dropout(p=p_drop),
            nn.Linear(64, 1)
        )
        
    def forward(self, x):
        return self.net(x)

model = ModelWithDropout(in_features = 20, p_drop = 0.5)
x = torch.randn(1, 20)

model.train()
out1 = model(x)
out2 = model(x)
print('Trining Mode - same input, different outputs:')
print(f'Call 1: {out1.item():.4f}')
print(f'Call 2: {out2.item():.4f}')
print(f'Same? {torch.allclose(out1, out2)}')

model.eval()
with torch.no_grad():
    out3 = model(x)
    out4 = model(x)
print('\nEval Mode - same input, identical outputs:')
print(f'Call 3: {out3.item():.4f}')
print(f'Call 4: {out4.item():.4f}')
print(f'Same? {torch.allclose(out3, out4)}')

Trining Mode - same input, different outputs:
Call 1: -0.0122
Call 2: -0.0372
Same? False

Eval Mode - same input, identical outputs:
Call 3: -0.0808
Call 4: -0.0808
Same? True


In [4]:
# Early Stopping Implementation
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

class EarlyStopping:
    def __init__(self, patience = 10, min_delta = 1e-4, path = 'checkpoint.pth', verbose = True):
        self.patience = patience
        self.min_delta = min_delta
        self.path = path
        self.verbose = verbose
        
        self.best_loss = float('inf')
        self.counter = 0  # epochs since last improvement
        self.should_stop = False  # flag the training loop checks

    def step(self, val_loss: float, model: 'nn.Module') -> bool:
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            self._save(model)
            if self.verbose:
                print(f'Val Loss Improved to {val_loss:.6f} - checkpoint saved.')
        else:
            self.counter += 1
            if self.verbose:
                print(f'No Improvement for {self.counter}/{self.patience} epochs '
                      f'best: {self.best_loss:.6f}')
            if self.counter >= self.patience:
                self.should_stop = True
                if self.verbose:
                    print(f'\nEarly Stopping triggered after {self.patience} epochs without improvement.')
        return self.should_stop

    def _save(self, model):
        torch.save(model.state_dict(), self.path)

    def load_best(self, model):
        model.load_state_dict(torch.load(self.path))
        return model


torch.manual_seed(42)

X_train = torch.randn(800, 10)
y_train = (X_train[:, 0] > 0).float().unsqueeze(1)
X_val = torch.randn(200, 10)
y_val = (X_val[:, 0] > 0).float().unsqueeze(1)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size = 64, shuffle = True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size = 128, shuffle = False)

model = nn.Sequential(
    nn.Linear(10, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 1)
)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr = 0.001)

early_stopping = EarlyStopping(patience = 7, min_delta = 1e-4, path = 'best_early_stop.pth', verbose = True)
for epoch in range(200):
    model.train()
    for bX, by in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(bX), by)
        loss.backward()
        optimizer.step()

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for bX, by in val_loader:
            val_loss += criterion(model(bX), by).item() * bX.size(0)
    val_loss /= len(val_loader.dataset)

    print(f'Epoch {epoch + 1:>3} | Val Loss: {val_loss:.6f}', end = ' ')

    if early_stopping.step(val_loss, model):
        break

early_stopping.load_best(model)
print('\n\nLoaded best model weights from checkpoint.')

Epoch   1 | Val Loss: 0.677610 Val Loss Improved to 0.677610 - checkpoint saved.
Epoch   2 | Val Loss: 0.634335 Val Loss Improved to 0.634335 - checkpoint saved.
Epoch   3 | Val Loss: 0.566434 Val Loss Improved to 0.566434 - checkpoint saved.
Epoch   4 | Val Loss: 0.475792 Val Loss Improved to 0.475792 - checkpoint saved.
Epoch   5 | Val Loss: 0.380556 Val Loss Improved to 0.380556 - checkpoint saved.
Epoch   6 | Val Loss: 0.298180 Val Loss Improved to 0.298180 - checkpoint saved.
Epoch   7 | Val Loss: 0.236662 Val Loss Improved to 0.236662 - checkpoint saved.
Epoch   8 | Val Loss: 0.192498 Val Loss Improved to 0.192498 - checkpoint saved.
Epoch   9 | Val Loss: 0.162313 Val Loss Improved to 0.162313 - checkpoint saved.
Epoch  10 | Val Loss: 0.143219 Val Loss Improved to 0.143219 - checkpoint saved.
Epoch  11 | Val Loss: 0.130852 Val Loss Improved to 0.130852 - checkpoint saved.
Epoch  12 | Val Loss: 0.119864 Val Loss Improved to 0.119864 - checkpoint saved.
Epoch  13 | Val Loss: 0.1096

In [5]:
# Data Augmentation for Tabular Data
import torch
def tabular_augmentation(x: torch.Tensor, noise_std: float = 0.05, dropout_rate: float = 0.1) -> torch.Tensor:
    noise = torch.randn_like(x) * noise_std
    x_aug = x + noise

    mask = (torch.rand_like(x) > dropout_rate).float()
    x_aug = x_aug * mask

    return x_aug

for bX, by in train_loader:
    bX_aug = tabular_augmentation(bX, noise_std = 0.05)
    optimizer.zero_grad()
    output = model(bX_aug)
    loss = criterion(output, by)
    loss.backwrd()
    optimizer.step()

AttributeError: 'Tensor' object has no attribute 'backwrd'

In [14]:
# Diagnosing and Fixing Real Overfitting Problem
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np

torch.manual_seed(42)
np.random.seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# 1. Small dataset tht easily overfits
n_train, n_val = 200, 100
n_features = 20

X_train = torch.randn(n_train, n_features)
X_val = torch.randn(n_val, n_features)

y_train = (X_train[:, 0] - X_train[:, 1] + 0.5 * X_train[:, 2] > 0).float().unsqueeze(1)
y_val = (X_val[:, 0] - X_val[:, 1] + 0.5 * X_val[:, 2] > 0).float().unsqueeze(1)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size = 32, shuffle = True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size = 64, shuffle = False)

# 2. Large Model (so that it overfits small dataset)
def model(use_dropout = False, dropout_p = 0.3):
    layers = [nn.Linear(n_features, 256), nn.ReLU(),]
    if use_dropout: layers.append(nn.Dropout(dropout_p))
    layers += [nn.Linear(256, 128), nn.ReLU(),]
    if use_dropout: layers.append(nn.Dropout(dropout_p))
    layers += [nn.Linear(128, 64), nn.ReLU(),]
    if use_dropout: layers.append(nn.Dropout(dropout_p))
    layers += [nn.Linear(64, 1)]

    return nn.Sequential(*layers).to(device)

# 3. Training Function
def train_and_evaluate(model, optimizer, n_epochs = 100, use_early_stopping = False):
    criterion = nn.BCEWithLogitsLoss()
    es = EarlyStopping(patience = 15, verbose = False) if use_early_stopping else None
    history = {'train':[], 'val':[]}

    for epoch in range(n_epochs):
        model.train()
        train_loss, n = 0.0, 0

        for bX, by in train_loader:
            bX, by = bX.to(device), by.to(device)
            optimizer.zero_grad()
            loss = criterion(model(bX), by)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * bX.size(0)
            n += bX.size(0)
        train_loss /= n


        model.eval()
        val_loss, n = 0.0, 0
        with torch.no_grad():
            for bX, by in val_loader:
                bX, by = bX.to(device), by.to(device)
                val_loss += criterion(model(bX), by).item() * bX.size(0)
                n += bX.size(0)
        val_loss /= n

        history['train'].append(train_loss)
        history['val'].append(val_loss)

        if es and es.step(val_loss, model):
            break

    return history, len(history['train'])

# 4. Experiment1 -- Overfitting baseline
print('Experiment 1: Large model, no regularization (Overfitting)')

model_overfit = model(use_dropout = False)
opt_overfit = optim.Adam(model_overfit.parameters(), lr = 0.001)
hist_overfit, epochs_run = train_and_evaluate(model_overfit, opt_overfit, n_epochs = 100)

final_train = hist_overfit['train'][-1]
final_val = hist_overfit['val'][-1]
best_val = min(hist_overfit['val'])
gap = final_val - final_train

print(f'After {epochs_run} epochs:')
print(f'Final Train Loss: {final_train:.4f}')
print(f'Final Val Loss: {final_val:.4f}')
print(f'Best Val Loss: {best_val:.4f}')
print(f'Train-Val Gap: {gap:.4f} <- large gap = overfitting')

# 5. Experiment2 -- L2 regularization
print('\n\nExperiment 2: L2 Regularization (weight_decay = 1e-3)')

model_l2 = model(use_dropout = False)
opt_l2 = optim.AdamW(model_l2.parameters(), lr = 0.001, weight_decay = 1e-3)
hist_l2, epochs_run = train_and_evaluate(model_l2, opt_l2, n_epochs = 100)

final_train = hist_l2['train'][-1]
final_val = hist_l2['val'][-1]
best_val = min(hist_l2['val'])
gap = final_val - final_train

print(f'After {epochs_run} epochs:')
print(f'Final Train Loss: {final_train:.4f}')
print(f'Final Val Loss: {final_val:.4f}')
print(f'Best Val Loss: {best_val:.4f}')
print(f'Train-Val Gap: {gap:.4f}')

# 6. Experiment3 -- Dropout
print('\n\nExperiment 3: Dropout (p = 0.4)')

model_dropout = model(use_dropout = True, dropout_p = 0.4)
opt_dropout = optim.Adam(model_dropout.parameters(), lr = 0.001)
hist_dropout, epochs_run = train_and_evaluate(model_dropout, opt_dropout, n_epochs = 100)

final_train = hist_dropout['train'][-1]
final_val = hist_dropout['val'][-1]
best_val = min(hist_dropout['val'])
gap = final_val - final_train

print(f'After {epochs_run} epochs:')
print(f'Final Train Loss: {final_train:.4f}')
print(f'Final Val Loss: {final_val:.4f}')
print(f'Best Val Loss: {best_val:.4f}')
print(f'Train-Val Gap: {gap:.4f}')

# 7. Experiment4 -- Combined Regularization
print('\n\nExperiment 4: L2 + Dropout + Early Stopping (combined)')

model_combined = model(use_dropout = True, dropout_p = 0.3)
opt_combined = optim.AdamW(model_combined.parameters(), lr = 0.001, weight_decay = 1e-3)
hist_combined, epochs_run = train_and_evaluate(model_combined, opt_combined, n_epochs = 200, 
                                               use_early_stopping = True)

final_train = hist_combined['train'][-1]
final_val = hist_combined['val'][-1]
best_val = min(hist_combined['val'])
gap = final_val - final_train

print(f'Stopped after {epochs_run} epochs (max was 200):')
print(f'Final Train Loss: {final_train:.4f}')
print(f'Final Val Loss: {final_val:.4f}')
print(f'Best Val Loss: {best_val:.4f}')
print(f'Train-Val Gap: {gap:.4f}')

# 8. Summary Comparison
print('\n\nSummary: Best Validation Loss Across Experiments')
results = {
    'No Regularization (overfit)' : min(hist_overfit['val']),
    'L2 only' : min(hist_l2['val']),
    'Dropout only' : min(hist_dropout['val']),
    'L2 + Dropout + Early Stop' : min(hist_combined['val']),
}
for name, val in results.items():
    print(f'{name:<38} {val:.4f}')

Experiment 1: Large model, no regularization (Overfitting)
After 100 epochs:
Final Train Loss: 0.0000
Final Val Loss: 0.4123
Best Val Loss: 0.2205
Train-Val Gap: 0.4122 <- large gap = overfitting


Experiment 2: L2 Regularization (weight_decay = 1e-3)
After 100 epochs:
Final Train Loss: 0.0000
Final Val Loss: 0.3882
Best Val Loss: 0.2293
Train-Val Gap: 0.3881


Experiment 3: Dropout (p = 0.4)
After 100 epochs:
Final Train Loss: 0.0006
Final Val Loss: 0.5397
Best Val Loss: 0.2191
Train-Val Gap: 0.5392


Experiment 4: L2 + Dropout + Early Stopping (combined)
Stopped after 26 epochs (max was 200):
Final Train Loss: 0.0215
Final Val Loss: 0.4543
Best Val Loss: 0.2682
Train-Val Gap: 0.4328


Summary: Best Validation Loss Across Experiments
No Regularization (overfit)            0.2205
L2 only                                0.2293
Dropout only                           0.2191
L2 + Dropout + Early Stop              0.2682


In [20]:
# Mini Exercise --> Effect of different reg techniques on overfitting problems
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# 1. Small Datasets
n_train, n_val = 300, 150
n_features = 30

X_train = torch.randn(n_train, n_features)
X_val = torch.randn(n_val, n_features)

# Only 3 of 30 features are informative
signal_train = X_train[:, 0] * 2.0 - X_train[:, 1] + X_train[:, 2] * 0.5
signal_val = X_val[:, 0] * 2.0 - X_val[:, 1] + X_val[:, 2] * 0.5

y_train = (signal_train > 0).float().unsqueeze(1)
y_val = (signal_val > 0).float().unsqueeze(1)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size = 32, shuffle = True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size = 64, shuffle = False)

# 2. Creating Model
def build_experiment_model(dropout_p = 0.0):
    layers = [nn.Linear(n_features, 512), nn.ReLU()]
    if dropout_p > 0: layers.append(nn.Dropout(dropout_p))
    layers += [nn.Linear(512, 256), nn.ReLU()]
    if dropout_p > 0: layers.append(nn.Dropout(dropout_p))
    layers += [nn.Linear(256, 128), nn.ReLU()]
    if dropout_p > 0: layers.append(nn.Dropout(dropout_p))
    layers += [nn.Linear(128, 1)]

    return nn.Sequential(*layers).to(device)


# 3. Helper fn to run one full training experiment
def run_experiment(model, optimizer, n_epochs = 100, patience = None, desc = ''):
    criterion = nn.BCEWithLogitsLoss()
    es = EarlyStopping(patience = patience, verbose = False, path = f'exp_{desc}.pth') if patience else None
    history = {'train' : [], 'val' : []}

    for epoch in range(n_epochs):
        model.train()
        tl, tn = 0.0, 0
        for bX, by in train_loader:
            bX, by = bX.to(device), by.to(device)
            optimizer.zero_grad()
            loss = criterion(model(bX), by)
            loss.backward()
            optimizer.step()

            tl += loss.item() * bX.size(0)
            tn += bX.size(0)

        model.eval()
        vl, vn = 0.0, 0
        with torch.no_grad():
            for bX, by in val_loader:
                bX, by = bX.to(device), by.to(device)
                optimizer.zero_grad()
                vl += criterion(model(bX), by).item() * bX.size(0)
                vn += bX.size(0)

        history['train'].append(tl / tn)
        history['val'].append(vl / vn)

        if es and es.step(vl / vn, model):
            break

    return history

# 4. Running the Experiment
configs = [
    ('Baseline (no regularization)', 0.0, 0.0, False, None),
    ('L2 only (weight_decay = 1e-3)', 0.0, 1e-3, True, None),
    ('Dropout only (p = 0.4)', 0.0, 0.0, False, None),
    ('Early Stopping only (patience = 10)', 0.0, 0.0, False, 10),
    ('L2 + Dropout + Early Stopping', 0.4, 1e-3, True, 10),
]
print(f"{'Configuration':<42} {'Best Val Loss':>14} {'Final Gap':>12}")

for i, (desc, dp, wd, use_adamw, patience) in enumerate(configs):
    model = build_experiment_model(dropout_p = dp)

    if use_adamw:
        opt = optim.AdamW(model.parameters(), lr = 0.001, weight_decay = wd)
    else: 
        opt = optim.Adam(model.parameters(), lr = 0.001)

    hist = run_experiment(model, opt, n_epochs = 100, patience = patience, desc = desc[:10])

    best_val = min(hist['val'])
    final_gap = hist['val'][-1] - hist['train'][-1]
    print(f'{i}. {desc:<42} {best_val:>14.4f} {final_gap:>12.4f}')

Configuration                               Best Val Loss    Final Gap
0. Baseline (no regularization)                       0.0802       0.1145
1. L2 only (weight_decay = 1e-3)                      0.0495       0.0637
2. Dropout only (p = 0.4)                             0.0872       0.1340
3. Early Stopping only (patience = 10)                0.1277       0.1324
4. L2 + Dropout + Early Stopping                      0.0835       0.1000
